In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from google.colab import files

# --- Configuración del Problema de Julián ---
# Restricción 1 (Materiales): 20x + 80y <= 160,000
# Restricción 2 (Personal):   50x + 80y <= 240,000
# Precios: Económica = 100, Lujo = 230

# Puntos de corte para el gráfico
# R1: (8000, 0) y (0, 2000)
# R2: (4800, 0) y (0, 3000)
# Intersección: 20x + 80y = 160000; 50x + 80y = 240000 -> 30x = 80000 -> x=2666.6, y=1333.3

precio_eco = 100
precios_lujo_var = np.linspace(100, 400, 60) # Variamos el precio de lujo para ver el cambio

fig, ax = plt.subplots(figsize=(10, 7))

def update(frame):
    ax.clear()
    p_lujo = precios_lujo_var[frame]
    
    # Rango de X (Batería Económica)
    x = np.linspace(0, 5000, 1000)
    
    # Ecuaciones de las restricciones (despejando y)
    r1 = (160000 - 20*x) / 80  # Materiales
    r2 = (240000 - 50*x) / 80  # Personal
    
    # Región Factible
    y_factible = np.minimum(np.maximum(0, r1), np.maximum(0, r2))
    ax.fill_between(x, 0, y_factible, color='palegreen', alpha=0.4, label="Región Factible (Producción)")
    
    # Dibujar líneas de restricción
    ax.plot(x, r1, color='blue', lw=2, label='Límite Materiales (€160k)')
    ax.plot(x, r2, color='red', lw=2, label='Límite Personal (€240k)')
    
    # Cálculo del Óptimo (Evaluamos vértices: (0,2000), (2666, 1333), (4800,0))
    vertices = [(0, 2000), (2666.67, 1333.33), (4800, 0)]
    beneficios = [precio_eco*v[0] + p_lujo*v[1] for v in vertices]
    idx_max = np.argmax(beneficios)
    x_opt, y_opt = vertices[idx_max]
    Z_max = beneficios[idx_max]
    
    # Dibujar Función Objetivo (Línea de nivel que pasa por el óptimo)
    # y = (Z - c1*x) / c2
    y_obj = (Z_max - precio_eco * x) / p_lujo
    ax.plot(x, y_obj, linestyle='--', color='darkgreen', lw=2, label=f'Z Máx: {Z_max:,.0f}€')
    
    # Marcar el punto óptimo
    ax.plot(x_opt, y_opt, 'ko', markersize=10)
    ax.annotate(f'ÓPTIMO\n({int(x_opt)}, {int(y_opt)})', (x_opt, y_opt), 
                xytext=(x_opt+200, y_opt+200), fontsize=10, fontweight='bold',
                arrowprops=dict(facecolor='black', shrink=0.05, width=1))

    # Estética
    ax.set_xlim(0, 5500)
    ax.set_ylim(0, 3500)
    ax.set_title(f"Optimización de Ventas: Precio Lujo = {p_lujo:.0f}€", fontsize=14)
    ax.set_xlabel("Baterías Económicas ($x_1$)")
    ax.set_ylabel("Baterías de Lujo ($x_2$)")
    ax.legend(loc='upper right')
    ax.grid(True, linestyle=':', alpha=0.6)

# Crear animación
ani = FuncAnimation(fig, update, frames=len(precios_lujo_var), interval=100)
nombre_archivo = "optimizacion_julian.gif"
ani.save(nombre_archivo, writer=PillowWriter(fps=10))

print(f"¡Listo! El archivo '{nombre_archivo}' se ha generado.")
files.download(nombre_archivo)